# D20 S1 A2 - One U-Net Down/Up Stage with NumPy
Real array operations for same convolution stand-in, 2x2 max pooling, nearest-neighbour upsampling, and skip concatenation.


In [ ]:
import numpy as np


In [ ]:
def same_conv(x, out_ch):
    x = np.asarray(x)
    mean_map = x.mean(axis=0)
    return np.broadcast_to(mean_map, (out_ch, *mean_map.shape)).copy()


def maxpool2x2(x):
    x = np.asarray(x)
    C, H, W = x.shape
    if H % 2 != 0 or W % 2 != 0:
        raise ValueError("Height and width must be even for 2x2 max pooling")
    blocks = x.reshape(C, H // 2, 2, W // 2, 2)
    return blocks.max(axis=(2, 4))


def upsample2x(x):
    x = np.asarray(x)
    return np.repeat(np.repeat(x, 2, axis=1), 2, axis=2)


def concat_skip(decoder, encoder):
    decoder = np.asarray(decoder)
    encoder = np.asarray(encoder)
    if decoder.shape[1:] != encoder.shape[1:]:
        raise ValueError("Decoder and encoder spatial dimensions must match")
    return np.concatenate([decoder, encoder], axis=0)


def down_up_with_skip(x, mid_ch, out_ch):
    skip = same_conv(x, mid_ch)
    pooled = maxpool2x2(skip)
    bottleneck = same_conv(pooled, out_ch)
    up = same_conv(upsample2x(bottleneck), mid_ch)
    merged = concat_skip(up, skip)
    final = same_conv(merged, mid_ch)
    return {
        "skip": skip,
        "pooled": pooled,
        "bottleneck": bottleneck,
        "up": up,
        "merged": merged,
        "final": final,
    }


In [ ]:
x = np.array([[[1,2,3,4],
               [5,6,7,8],
               [9,10,11,12],
               [13,14,15,16]]], dtype=float)
print(maxpool2x2(x))
print(upsample2x(np.array([[[1,2],[3,4]]], dtype=float)).shape)

x = np.random.RandomState(0).rand(1, 64, 64)
tr = down_up_with_skip(x, 8, 16)
print({k: v.shape for k, v in tr.items()})
print(np.array_equal(tr["merged"][:8], tr["up"]),
      np.array_equal(tr["merged"][8:], tr["skip"]))


In [ ]:
# Checks corresponding to all 8 autograder requirements
sample = np.arange(2 * 3 * 4, dtype=float).reshape(2, 3, 4)
sc = same_conv(sample, 5)
expected_mean = sample.mean(axis=0)
assert sc.shape == (5, 3, 4)
assert np.all(sc == expected_mean)

pool_input = np.array([[[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]], dtype=float)
assert np.array_equal(maxpool2x2(pool_input), np.array([[[6,8],[14,16]]], dtype=float))

small = np.array([[[1,2],[3,4]]], dtype=float)
expected_up = np.array([[[1,1,2,2],[1,1,2,2],[3,3,4,4],[3,3,4,4]]], dtype=float)
assert np.array_equal(upsample2x(small), expected_up)

dec = np.ones((2, 4, 4))
enc = np.full((3, 4, 4), 7.0)
merged_test = concat_skip(dec, enc)
assert np.array_equal(merged_test[:2], dec)
assert np.array_equal(merged_test[2:], enc)

assert tr["pooled"].shape == (8, 32, 32)
assert tr["bottleneck"].shape == (16, 32, 32)
assert tr["merged"].shape == (16, 64, 64)
assert tr["final"].shape == (8, 64, 64)
assert np.array_equal(tr["merged"][:8], tr["up"])
assert np.array_equal(tr["merged"][8:], tr["skip"])

try:
    concat_skip(np.zeros((2,4,4)), np.zeros((2,5,4)))
    raise AssertionError("concat_skip should raise ValueError")
except ValueError:
    pass

special = np.array([[[0,0],[0,9]]], dtype=float)
assert maxpool2x2(special)[0,0,0] == 9
for odd in (np.zeros((1,3,4)), np.zeros((1,4,3))):
    try:
        maxpool2x2(odd)
        raise AssertionError("maxpool2x2 should raise ValueError for odd dimensions")
    except ValueError:
        pass

print("All 8 autograder-style checks passed.")
